# Standalone edition
This notebook embeds its reusable Python utilities and can be opened independently. The primary demonstration is a small, transparent Transformer; the additional lifecycle and LSTM sections extend the same engine without creating a competing notebook. Install PyTorch and NumPy in the notebook environment before running.

**Version 3 focus:** Weight Space Programming + machine unlearning assurance + security-native SDLC + event-driven zero-trust assessment.

# 02 — Weight Space Programming & Security-Native Model Lifecycle Engine
### The central programmable weight-space engine with continuous, zero-trust assurance

**Primary demonstration architecture:** the existing small decoder-only PyTorch Transformer (2 layers, width 128, 4-head causal self-attention, SwiGLU, RMSNorm, pre-norm residuals, tied token/output embeddings). It remains the primary executable demonstration. An LSTM adapter is added as a complementary research fixture for explicit memory-gate analysis.

**Purpose:** inspect, compare, probe, and safely experiment with neural parameter spaces while treating security as a parallel, mandatory SDLC workstream. The engine covers model changes and the surrounding application lifecycle: modernization, digitalization, containerization, CI/CD, deployment, runtime change, scaling, maintenance, and end-of-life.

**Operating rule:** source artifacts are read-only; candidate changes are isolated; each material change is tied to provenance, security evidence, behavioral validation, policy authorization, and a rollback reference. This notebook provides local assurance contracts and demonstration logic—not a claim that it replaces specialist scanners, penetration testing, or production authorization services.

In [1]:
import copy
import math
import json
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(11)

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight

class CausalSelfAttention(nn.Module):
    def __init__(self, dim=128, heads=4):
        super().__init__()
        assert dim % heads == 0
        self.dim, self.heads, self.head_dim = dim, heads, dim // heads
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)
    def forward(self, x):
        b, t, d = x.shape
        def split_heads(z):
            return z.view(b, t, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split_heads, (self.q_proj(x), self.k_proj(x), self.v_proj(x)))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(b, t, d)
        return self.o_proj(context)

class SwiGLU(nn.Module):
    def __init__(self, dim=128, hidden_dim=256):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, dim=128, heads=4, mlp_dim=256):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, heads)
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, mlp_dim)
    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x

class MiniDecoderTransformer(nn.Module):
    """Small decoder-only LM for architecture-aware experiments; not pretrained."""
    def __init__(self, vocab_size=32, max_seq_len=16, dim=128, heads=4,
                 layers=2, mlp_dim=256, tie_embeddings=True):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, max_seq_len=max_seq_len,
                          dim=dim, heads=heads, layers=layers, mlp_dim=mlp_dim,
                          tie_embeddings=tie_embeddings)
        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, heads, mlp_dim) for _ in range(layers)
        ])
        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight
    def forward(self, input_ids):
        b, t = input_ids.shape
        if t > self.config['max_seq_len']:
            raise ValueError('Sequence exceeds configured max_seq_len')
        positions = torch.arange(t, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))

def permute_attention_heads_(model, layer_index, permutation):
    """Function-preserving head permutation for this explicit MHA implementation."""
    attn = model.blocks[layer_index].attn
    p = torch.as_tensor(permutation, dtype=torch.long)
    if sorted(p.tolist()) != list(range(attn.heads)):
        raise ValueError('permutation must contain each attention head exactly once')
    idx = torch.cat([torch.arange(int(h)*attn.head_dim, (int(h)+1)*attn.head_dim) for h in p]).to(attn.q_proj.weight.device)
    with torch.no_grad():
        for proj in (attn.q_proj, attn.k_proj, attn.v_proj):
            proj.weight.copy_(proj.weight[idx, :])
        attn.o_proj.weight.copy_(attn.o_proj.weight[:, idx])
    return model

def permute_swiglu_units_(model, layer_index, permutation):
    """Function-preserving permutation of the paired SwiGLU intermediate units."""
    ffn = model.blocks[layer_index].ffn
    p = torch.as_tensor(permutation, dtype=torch.long)
    if sorted(p.tolist()) != list(range(ffn.gate_proj.out_features)):
        raise ValueError('permutation must contain every SwiGLU intermediate unit')
    with torch.no_grad():
        ffn.gate_proj.weight.copy_(ffn.gate_proj.weight[p])
        ffn.up_proj.weight.copy_(ffn.up_proj.weight[p])
        ffn.down_proj.weight.copy_(ffn.down_proj.weight[:, p])
    return model

@torch.no_grad()
def ablate_attention_head(model, layer_index, head_index):
    """Ablate one head's contribution by zeroing its input slice to O projection."""
    candidate = copy.deepcopy(model)
    attn = candidate.blocks[layer_index].attn
    if not 0 <= head_index < attn.heads:
        raise IndexError('head_index out of range')
    start, end = head_index * attn.head_dim, (head_index + 1) * attn.head_dim
    attn.o_proj.weight[:, start:end].zero_()
    return candidate

@torch.no_grad()
def ablate_swiglu_unit(model, layer_index, unit_index):
    candidate = copy.deepcopy(model)
    ffn = candidate.blocks[layer_index].ffn
    if not 0 <= unit_index < ffn.gate_proj.out_features:
        raise IndexError('unit_index out of range')
    ffn.gate_proj.weight[unit_index].zero_()
    ffn.up_proj.weight[unit_index].zero_()
    ffn.down_proj.weight[:, unit_index].zero_()
    return candidate


def state_dict_sha256(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        digest.update(name.encode("utf-8"))
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def next_token_loss(model, input_ids, target_ids):
    logits = model(input_ids)
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))

model = MiniDecoderTransformer().to(DEVICE).eval()
print('Model hash:', state_dict_sha256(model))
print('Configuration:', model.config)
print('Tied embedding storage:', model.lm_head.weight.data_ptr() == model.token_embedding.weight.data_ptr())


Model hash: 01557e0375b799f256e856bf6563781698d3b7b2f6e9e10259472d91e43b11ca
Configuration: {'vocab_size': 32, 'max_seq_len': 16, 'dim': 128, 'heads': 4, 'layers': 2, 'mlp_dim': 256, 'tie_embeddings': True}
Tied embedding storage: True


## 1. Inspector: architecture and tensor inventory

Statistics are descriptive, not direct evidence that a tensor is useful or harmful. The architecture summary also records the model configuration and detects shared parameter storage, which matters for tied embeddings and safe transformations.


In [3]:
class WeightSpaceInspector:
    def __init__(self, model: nn.Module):
        self.model = model
    def architecture(self) -> Dict[str, Any]:
        shared = {}
        seen = {}
        for name, parameter in self.model.named_parameters(remove_duplicate=False):
            key = (parameter.untyped_storage().data_ptr(), parameter.storage_offset(), tuple(parameter.shape))
            if key in seen:
                shared.setdefault(seen[key], []).append(name)
            else:
                seen[key] = name
        return {
            'class': self.model.__class__.__name__,
            'config': getattr(self.model, 'config', None),
            'modules': [{'name': n, 'class': m.__class__.__name__} for n, m in self.model.named_modules()],
            'parameter_count': sum(p.numel() for p in self.model.parameters()),
            'trainable_parameter_count': sum(p.numel() for p in self.model.parameters() if p.requires_grad),
            'shared_parameter_aliases': shared,
        }
    def tensor_inventory(self):
        rows = []
        for name, tensor in self.model.state_dict().items():
            t = tensor.detach().float().cpu()
            rows.append({'name': name, 'shape': list(tensor.shape), 'dtype': str(tensor.dtype),
                         'numel': tensor.numel(), 'mean': t.mean().item() if t.numel() else 0.0,
                         'std': t.std(unbiased=False).item() if t.numel() else 0.0,
                         'min': t.min().item() if t.numel() else 0.0,
                         'max': t.max().item() if t.numel() else 0.0})
        return rows

inspector = WeightSpaceInspector(model)
print(json.dumps(inspector.architecture(), indent=2))
print('Tensor inventory preview:', json.dumps(inspector.tensor_inventory()[:8], indent=2))

{
  "class": "MiniDecoderTransformer",
  "config": {
    "vocab_size": 32,
    "max_seq_len": 16,
    "dim": 128,
    "heads": 4,
    "layers": 2,
    "mlp_dim": 256,
    "tie_embeddings": true
  },
  "modules": [
    {
      "name": "",
      "class": "MiniDecoderTransformer"
    },
    {
      "name": "token_embedding",
      "class": "Embedding"
    },
    {
      "name": "position_embedding",
      "class": "Embedding"
    },
    {
      "name": "blocks",
      "class": "ModuleList"
    },
    {
      "name": "blocks.0",
      "class": "TransformerBlock"
    },
    {
      "name": "blocks.0.attn_norm",
      "class": "RMSNorm"
    },
    {
      "name": "blocks.0.attn",
      "class": "CausalSelfAttention"
    },
    {
      "name": "blocks.0.attn.q_proj",
      "class": "Linear"
    },
    {
      "name": "blocks.0.attn.k_proj",
      "class": "Linear"
    },
    {
      "name": "blocks.0.attn.v_proj",
      "class": "Linear"
    },
    {
      "name": "blocks.0.attn.o_proj",
    

## 2. Functional comparison

A comparison needs representative token sequences. Agreement on a small probe set is evidence only for those probes, not proof of global equivalence.


In [4]:
@torch.no_grad()
def compare_outputs(model_a, model_b, probe_x):
    old_a, old_b = model_a.training, model_b.training
    try:
        model_a.eval(); model_b.eval()
        out_a, out_b = model_a(probe_x), model_b(probe_x)
        if out_a.shape != out_b.shape:
            raise ValueError('Output shape mismatch')
        return {'max_abs_error': (out_a-out_b).abs().max().item(),
                'mean_abs_error': (out_a-out_b).abs().mean().item(),
                'token_prediction_agreement': (out_a.argmax(-1) == out_b.argmax(-1)).float().mean().item()}
    finally:
        model_a.train(old_a); model_b.train(old_b)

probe_x = torch.randint(0, model.config['vocab_size'], (16, 12), device=DEVICE)
comparison = compare_outputs(model, copy.deepcopy(model), probe_x)
comparison

{'max_abs_error': 0.0,
 'mean_abs_error': 0.0,
 'token_prediction_agreement': 1.0}

## 3. Gradient sensitivity

Mean absolute gradient is a local, objective- and dataset-dependent diagnostic. Near-zero gradients do not prove irrelevance. Here the objective is next-token prediction over synthetic token sequences.


In [5]:
def gradient_sensitivity(model, input_ids, target_ids):
    was_training = model.training
    model.zero_grad(set_to_none=True)
    try:
        model.train()
        logits = model(input_ids)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))
        loss.backward()
        report = {}
        for name, parameter in model.named_parameters():
            if parameter.grad is None:
                report[name] = {'mean_abs_grad': None, 'max_abs_grad': None}
            else:
                grad = parameter.grad.detach().abs()
                report[name] = {'mean_abs_grad': grad.mean().item(), 'max_abs_grad': grad.max().item()}
        return loss.item(), report
    finally:
        model.zero_grad(set_to_none=True)
        model.train(was_training)

x = torch.randint(0, model.config['vocab_size'], (12, 12), device=DEVICE)
y = torch.randint(0, model.config['vocab_size'], (12, 12), device=DEVICE)
loss_value, sensitivity = gradient_sensitivity(model, x, y)
print('Next-token probe loss:', loss_value)
print(json.dumps({k:v for k,v in sensitivity.items() if 'blocks.0.attn' in k}, indent=2))

Next-token probe loss: 83.69979858398438
{
  "blocks.0.attn_norm.weight": {
    "mean_abs_grad": 0.018299221992492676,
    "max_abs_grad": 0.06837578862905502
  },
  "blocks.0.attn.q_proj.weight": {
    "mean_abs_grad": 0.0049506318755447865,
    "max_abs_grad": 0.036083679646253586
  },
  "blocks.0.attn.k_proj.weight": {
    "mean_abs_grad": 0.005200645420700312,
    "max_abs_grad": 0.04029799625277519
  },
  "blocks.0.attn.v_proj.weight": {
    "mean_abs_grad": 0.025280006229877472,
    "max_abs_grad": 0.19548611342906952
  },
  "blocks.0.attn.o_proj.weight": {
    "mean_abs_grad": 0.02506684698164463,
    "max_abs_grad": 0.2122647911310196
  }
}


## 4. Counterfactual intervention: ablate one attention head

This operation is architecture-aware for the explicit `MiniDecoderTransformer`: it zeros the selected head's output-channel slice entering the O projection, on a deep-copied candidate. This is an intervention, not a function-preserving transformation.


In [6]:
def evaluate_loss(model, input_ids, target_ids):
    model.eval()
    with torch.no_grad():
        logits = model(input_ids)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1)).item()

baseline_loss = evaluate_loss(model, x, y)
ablation_rows = []
for head in range(model.blocks[0].attn.heads):
    candidate = ablate_attention_head(model, layer_index=0, head_index=head)
    altered_loss = evaluate_loss(candidate, x, y)
    ablation_rows.append({'layer': 0, 'attention_head': head,
                          'baseline_loss': baseline_loss, 'ablated_loss': altered_loss,
                          'delta_loss': altered_loss-baseline_loss})
sorted(ablation_rows, key=lambda r: r['delta_loss'])

[{'layer': 0,
  'attention_head': 3,
  'baseline_loss': 83.69979858398438,
  'ablated_loss': 83.74591064453125,
  'delta_loss': 0.046112060546875},
 {'layer': 0,
  'attention_head': 0,
  'baseline_loss': 83.69979858398438,
  'ablated_loss': 83.76478576660156,
  'delta_loss': 0.0649871826171875},
 {'layer': 0,
  'attention_head': 2,
  'baseline_loss': 83.69979858398438,
  'ablated_loss': 83.89557647705078,
  'delta_loss': 0.19577789306640625},
 {'layer': 0,
  'attention_head': 1,
  'baseline_loss': 83.69979858398438,
  'ablated_loss': 83.9063491821289,
  'delta_loss': 0.20655059814453125}]

## 5. Symmetry-preserving Transformer permutations

The adapter supports two explicit transformations: attention-head permutation (Q/K/V output blocks plus O input blocks) and SwiGLU intermediate-unit permutation (paired gate/up rows plus down-projection columns). It intentionally does not claim compatibility with arbitrary Transformer implementations, GQA/MQA, fused QKV layouts, rotary variants, or third-party checkpoint conventions.


In [7]:
permuted_model = copy.deepcopy(model)
permute_attention_heads_(permuted_model, layer_index=0, permutation=[2, 0, 3, 1])
symmetry_check = compare_outputs(model, permuted_model, probe_x)
print('Attention-head permutation:', symmetry_check)
assert symmetry_check['max_abs_error'] < 1e-4

swiglu_model = copy.deepcopy(model)
permute_swiglu_units_(swiglu_model, 0, torch.randperm(model.blocks[0].ffn.gate_proj.out_features))
swiglu_check = compare_outputs(model, swiglu_model, probe_x)
print('SwiGLU permutation:', swiglu_check)
assert swiglu_check['max_abs_error'] < 1e-4

Attention-head permutation: {'max_abs_error': 3.814697265625e-05, 'mean_abs_error': 1.5222276488202624e-06, 'token_prediction_agreement': 1.0}
SwiGLU permutation: {'max_abs_error': 2.288818359375e-05, 'mean_abs_error': 1.3227257795733749e-06, 'token_prediction_agreement': 1.0}


## 6. Auditable report export

The report stores measurements and model hashes. It does not claim a component is globally irrelevant and does not modify the source checkpoint.

In [8]:
def build_report(model, inspector, comparison, sensitivity, ablations):
    return {
        "engine_version": "0.1.0",
        "model_sha256": state_dict_sha256(model),
        "architecture": inspector.architecture(),
        "tensor_inventory": inspector.tensor_inventory(),
        "functional_comparison": comparison,
        "gradient_sensitivity": sensitivity,
        "counterfactual_ablation": ablations,
        "limitations": [
            "Results are conditional on the supplied probe data and objective.",
            "Low gradient or low variance does not establish parameter irrelevance.",
            "Architecture-specific transformations require explicit compatibility checks.",
        ],
    }

report = build_report(model, inspector, symmetry_check, sensitivity, ablation_rows)
Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/weight_space_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Wrote artifacts/weight_space_report.json")

Wrote artifacts/weight_space_report.json


## 7. Optional NFN research integration

The [AllanYangZhou/nfn repository](https://github.com/AllanYangZhou/nfn) is a research implementation worth studying for permutation-equivariant weight-space processing. Treat it as an optional backend, not a universal checkpoint reader.

Before integrating:
1. Pin a commit and record its license/dependencies.
2. Run its own tests.
3. Inspect supported `network_spec` structures and tensor ordering.
4. Start with supported MLP/CNN examples.
5. Do not feed arbitrary Transformer state dictionaries into its helpers without an architecture-specific adapter.

Useful reference: [NFN test_layers.py](https://github.com/AllanYangZhou/nfn/blob/main/tests/test_layers.py).

## Engine safety contract

- Never overwrite the source model during experiments.
- Save candidate artifacts separately and record hashes before and after edits.
- Record the exact architecture adapter and transformation implementation version.
- Verify symmetry-preserving operations with output-level probes and numerical tolerances.
- Treat ablation as a scoped experiment, not proof of global irrelevance.
- Fail closed for unsupported architectures or parameter layouts.
- Keep evaluation dataset/version, runtime versions, and approval evidence in the report.


## 8. Product architecture: Model Intelligence & Assurance Engine

The engine should support several applications through one shared analysis core.

### Shared engine services

1. **Artifact intake:** PyTorch checkpoints, Safetensors, Hugging Face model metadata, hashes, provenance.
2. **Architecture adapters:** explicit tensor maps for supported MLP, CNN, and Transformer families.
3. **Weight-space representation:** tensor statistics, structured features, embeddings, optional NFN backend.
4. **Behavioral probing:** fixed, versioned input suites; optional hidden-state probes where supported.
5. **Counterfactual analysis:** ablations, controlled perturbations, sensitivity and group interventions.
6. **Security analysis:** integrity drift, suspicious structural patterns, provenance gaps, anomaly triage.
7. **Evidence reporting:** model identity, hashes, data versions, methods, findings, confidence and limitations.

### Adapter contract

Do not infer arbitrary tensor meaning from tensor names alone. Each architecture adapter must declare:
- model family and supported versions
- parameter names and shapes
- layer / head / neuron mappings
- tied-weight relationships
- valid symmetry transformations
- intervention and reconstruction rules
- known unsupported features

This is the foundation for safely extending from small MLP demonstrations to real Transformer checkpoints.

## 9. Product demo A — Model Artifact Security Assessment

**Customer problem:** a team receives a model checkpoint from a vendor, open-source repository, internal training run, or fine-tuning pipeline. They need evidence about what they received before deployment.

### Demo workflow
1. Load the artifact and metadata.
2. Calculate cryptographic hashes and record provenance.
3. Inspect architecture, tensor shapes, parameter counts and numerical statistics.
4. Compare against a trusted baseline where available.
5. Run structural anomaly heuristics and optional learned weight-space diagnostics.
6. Run a declared behavioral probe suite.
7. Produce a finding report with evidence and uncertainty.

### Example report fields
- Artifact identity and hash
- Architecture adapter and coverage
- Unexpected tensor / shape differences
- Parameter distribution shifts
- Probe-suite results
- Suspected anomaly indicators
- False-positive caveats
- Recommended next tests

**Security boundary:** anomaly detection is triage, not proof of a backdoor. No automatic blocking based solely on a statistical outlier.

## 10. Product demo B — Model Change Impact & Optimization

**Customer problem:** a model has been fine-tuned, merged, quantized, compressed, or edited. The team wants to know whether the change matters and whether it is safe to promote.

### Demo workflow
1. Compare source and candidate hashes and metadata.
2. Align models only when the architecture and symmetry assumptions permit it.
3. Compare raw parameter differences and functional outputs separately.
4. Run layer / head / neuron interventions on supported architectures.
5. Measure loss, task metrics, latency, memory, and regression-suite behavior.
6. Generate a promotion recommendation with supporting evidence.

### Product outputs
- Change-impact report
- Candidate redundancy hypotheses
- Capability-specific sensitivity map
- Performance / memory comparison
- Regression results
- Rollback reference

The engine must not equate low gradient, low variance, or low impact on one dataset with global irrelevance.

## 11. Product demo C — Model Discovery & Selection

**Customer problem:** an organization has many models and checkpoints but lacks a consistent way to identify which are appropriate for a task.

Potential research integrations:
- WeightCLIP-style joint model / dataset embeddings.
- Functional probe fingerprints.
- Architecture and provenance metadata.
- Task-specific benchmark profiles.
- Similarity search over previously assessed artifacts.

A model should not be selected solely because its weight embedding is close to a dataset embedding. The result is a candidate ranking that must be verified with task-specific evaluation.

## 12. Product demo D — Model Optimization Workbench

Potential integrations:
- NFN / equivariant weight-space modules for supported architectures.
- LoRA and conventional fine-tuning baselines.
- Compression and structured factorization.
- Candidate weight edits with immutable originals.
- Generative weight methods such as DeepWeightFlow as a research track.

The workbench compares methods against the same baseline and evaluation contract. It should report quality, latency, memory, artifact size, cost estimate, and safety regressions.

### Engine API direction

The eventual Python package should expose stable operations such as:
- `inspect_artifact(...)`
- `fingerprint_model(...)`
- `compare_models(...)`
- `run_probe_suite(...)`
- `analyze_sensitivity(...)`
- `assess_security_posture(...)`
- `evaluate_candidate_edit(...)`
- `export_evidence_report(...)`

These are interface targets; implement each only when its supported scope and tests are defined.

## 14. Activation functions, gradient flow and weight updates

Activation functions influence the gradients that reach trainable parameters. This section connects the earlier sigmoid/softmax discussion directly to Weight Space Programming.

For a parameter `w`, backpropagation applies the chain rule:

`dL/dw = dL/da * da/dz * dz/dw`

A saturated sigmoid or tanh can reduce local derivatives; ReLU can produce zero gradients for negative pre-activations; GELU and SiLU have smoother nonlinear responses. Softmax is different: it normalizes a vector and has a Jacobian `diag(p) - p p^T`, coupling class/token probabilities rather than acting independently on each logit.

The experiment below profiles derivatives on a fixed input grid and calculates a softmax Jacobian. These are local mathematical diagnostics, not standalone predictions of training quality. Actual weight updates also depend on loss, data, architecture, normalization, optimizer and learning rate.

In [ ]:
# Local derivative profiles for common activations.
activation_grid = torch.linspace(-8, 8, 801, device=DEVICE)
activation_fns = {
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "relu": F.relu,
    "gelu": F.gelu,
    "silu": F.silu,
}
activation_profile = {}
for name, fn in activation_fns.items():
    x = activation_grid.detach().clone().requires_grad_(True)
    y = fn(x)
    grad = torch.autograd.grad(y.sum(), x)[0]
    activation_profile[name] = {
        "mean_abs_derivative": float(grad.abs().mean()),
        "max_abs_derivative": float(grad.abs().max()),
        "near_zero_derivative_fraction": float((grad.abs() < 1e-3).float().mean()),
    }

# Softmax couples all positions/classes in a vector through its Jacobian.
softmax_logits = torch.tensor([2.0, 1.0, 0.1], device=DEVICE, requires_grad=True)
softmax_probs = torch.softmax(softmax_logits, dim=-1)
softmax_jacobian = torch.diag(softmax_probs) - torch.outer(softmax_probs, softmax_probs)

print("Activation derivative profiles:")
print(json.dumps(activation_profile, indent=2))
print("Softmax probabilities:", softmax_probs.detach().cpu().tolist())
print("Softmax Jacobian:\n", softmax_jacobian.detach().cpu())
assert torch.allclose(softmax_probs.sum(), torch.tensor(1.0, device=DEVICE), atol=1e-6)
assert torch.allclose(softmax_jacobian.sum(dim=-1), torch.zeros(3, device=DEVICE), atol=1e-6)

## 14. Weight Space Programming as a security-native SDLC capability

The engine is not only a tensor-inspection utility. It is a controlled engineering capability that accompanies an application from modernization through end-of-life.

### Parallel engineering tracks

**Application lifecycle:** discover → assess legacy application → modernize/digitalize → build → containerize → provision infrastructure → integrate APIs/models → deploy → operate/scale → maintain → retire.

**Security lifecycle (runs in parallel):** asset/attack-surface inventory → threat model → SAST/SCA/secrets/IaC/container/API/ML assessment → policy evaluation → signed build provenance → deployment authorization → runtime monitoring → event-triggered reassessment → decommissioning and data/artifact disposition.

Security is not a final checklist. Every lifecycle stage produces security evidence and has explicit acceptance criteria. Pipeline hooks and runtime events trigger near-real-time reassessment of the affected artifact and its dependencies; this is event-driven assurance, not a claim of instantaneous omniscience. No scanner can prove an application “vulnerability free”; the objective is measurable risk reduction, verified controls, and continuously updated assurance.

### Change events that trigger reassessment

| Event | Minimum reassessment |
|---|---|
| Source or dependency change | SAST, SCA, secret scanning, tests, SBOM refresh |
| Container rebuild/base-image change | Image scan, SBOM, configuration and provenance checks |
| IaC/Kubernetes/API change | IaC, admission/RBAC/network/API security checks |
| Model/checkpoint/adapter change | Hash/provenance, architecture, tensor diff, behavioral probes |
| Fine-tuning, weight edit, quantization or unlearning | Model integrity, retain/forget tests, safety regression, approval |
| Agent tool or permission change | Tool inventory, authorization and prompt-injection boundary tests |
| Runtime/configuration/scale change | Runtime posture, identity, network, resource and policy checks |
| End-of-life | Disable identities/endpoints, revoke credentials, archive evidence, dispose of data/artifacts |

## 15. Zero-trust assessment contract and release gates

Zero trust is applied to artifacts, identities, pipelines, model changes, and deployment decisions: verify explicitly, grant least privilege, assume breach, and re-evaluate when context changes.

This notebook defines a **fail-closed evidence contract**. External tools (for example Semgrep, Bandit, pip-audit, Gitleaks, Checkov, Trivy, kube-linter, API security tests, and model-specific evaluations) must produce versioned results that are ingested into this contract. The code below does not pretend to run those scanners itself.

A release decision requires evidence that is present, passing, current for the exact artifact, and attributable to an authorized assessment. Missing evidence is not a pass.

In [ ]:
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from typing import List, Optional

REQUIRED_SECURITY_CHECKS = {
    "source_sast", "dependency_sca", "secret_scan", "iac_scan",
    "container_scan", "sbom_provenance", "identity_policy",
    "runtime_or_deployment_policy", "behavioral_regression"
}

@dataclass
class SecurityEvidence:
    check_id: str
    artifact_sha256: str
    status: str                 # "pass", "fail", or "not_run"
    tool: str
    tool_version: str
    assessed_at_utc: str
    assessor: str
    evidence_ref: str
    notes: str = ""

@dataclass
class ChangeEvent:
    event_id: str
    artifact_id: str
    artifact_type: str          # application, container, iac, model, dataset, agent, etc.
    change_type: str
    lifecycle_stage: str
    before_sha256: Optional[str]
    after_sha256: str
    requested_by: str
    authorized_by: Optional[str] = None
    change_ticket: Optional[str] = None

@dataclass
class ReleaseDecision:
    decision: str               # approve, reject, quarantine
    artifact_sha256: str
    missing_checks: List[str]
    failed_checks: List[str]
    stale_checks: List[str]
    reasons: List[str]


def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def evaluate_zero_trust_gate(artifact_sha256: str,
                             evidence: List[SecurityEvidence],
                             required_checks=REQUIRED_SECURITY_CHECKS,
                             max_age_hours: int = 24,
                             authorized: bool = False) -> ReleaseDecision:
    """Fail closed unless exact-artifact evidence is complete, passing and fresh."""
    now = datetime.now(timezone.utc)
    by_check = {item.check_id: item for item in evidence
                if item.artifact_sha256 == artifact_sha256}
    missing, failed, stale, reasons = [], [], [], []
    for check in sorted(required_checks):
        item = by_check.get(check)
        if item is None or item.status == "not_run":
            missing.append(check)
            continue
        if item.status != "pass":
            failed.append(check)
        try:
            age_hours = (now - datetime.fromisoformat(item.assessed_at_utc)).total_seconds() / 3600
            if age_hours < 0 or age_hours > max_age_hours:
                stale.append(check)
        except Exception:
            stale.append(check)
        if not all([item.tool, item.tool_version, item.assessor, item.evidence_ref]):
            reasons.append(f"{check}: incomplete evidence provenance")
    if not authorized:
        reasons.append("Independent release authorization is missing")
    if missing or failed or stale or reasons:
        return ReleaseDecision("quarantine", artifact_sha256, missing, failed, stale, reasons)
    return ReleaseDecision("approve", artifact_sha256, [], [], [], [])


def required_checks_for_event(event: ChangeEvent) -> set:
    """Return additional checks triggered by a material change event."""
    # Shared controls apply to every artifact; specialist checks depend on its attack surface.
    base = {"artifact_integrity", "provenance", "identity_policy", "authorization", "audit_record"}
    kind = event.artifact_type.lower()
    change = event.change_type.lower()
    if kind == "application" or kind == "source":
        base |= {"source_sast", "dependency_sca", "secret_scan", "unit_integration_tests", "api_security_tests"}
    if kind == "container" or "container" in change:
        base |= {"image_vulnerability_scan", "sbom_generation", "image_configuration", "runtime_configuration"}
    if kind in {"iac", "infrastructure", "kubernetes"}:
        base |= {"iac_scan", "kubernetes_policy", "rbac_network_policy"}
    if kind == "model" or any(x in change for x in ("weight", "fine_tune", "unlearn", "checkpoint", "quantiz")):
        base |= {"checkpoint_integrity", "architecture_validation", "weight_diff", "model_provenance",
                 "behavioral_regression", "safety_regression"}
        if "unlearn" in change:
            base |= {"retain_set_eval", "forget_set_eval", "generalization_eval", "reference_model_comparison"}
    if kind in {"agent", "tool"} or "tool_permission" in change:
        base |= {"tool_inventory", "tool_authorization_tests", "prompt_injection_boundary_tests"}
    if kind == "dataset" or "dataset" in change:
        base |= {"dataset_provenance", "data_access_policy", "poisoning_quality_checks"}
    if event.lifecycle_stage.lower() in {"end_of_life", "retirement", "decommission"}:
        base |= {"credential_revocation", "endpoint_decommission", "data_disposition", "evidence_archive"}
    return base

print("Required baseline checks:", len(REQUIRED_SECURITY_CHECKS))
print("Zero-trust gate ready; missing evidence will quarantine by default.")

In [ ]:
# Small executable policy tests: missing evidence and stale evidence must never pass.
_test_hash = "a" * 64
_empty_decision = evaluate_zero_trust_gate(_test_hash, [], authorized=True)
assert _empty_decision.decision == "quarantine"
assert set(_empty_decision.missing_checks) == REQUIRED_SECURITY_CHECKS

_demo_event = ChangeEvent(
    event_id="evt-demo-001", artifact_id="demo-model", artifact_type="model",
    change_type="unlearning", lifecycle_stage="model_update",
    before_sha256="b" * 64, after_sha256=_test_hash,
    requested_by="researcher", authorized_by=None, change_ticket="WSP-DEMO-1"
)
_model_checks = required_checks_for_event(_demo_event)
assert {"checkpoint_integrity", "retain_set_eval", "forget_set_eval", "safety_regression"}.issubset(_model_checks)
assert "source_sast" not in _model_checks  # model artifact checks follow its own attack surface
print("Policy contract tests passed. Model-change checks:", sorted(_model_checks))

## 16. Model-change assurance: weights, behavior, unlearning and provenance

A model can keep the same API and container while its learned parameters change. Therefore, a model update is a security-relevant application change even when application source code is unchanged.

### Mandatory model-change evidence

1. Original and candidate checkpoint hashes; immutable source reference.
2. Model identity, architecture adapter/version, configuration, tokenizer and dependency versions.
3. Tensor inventory and architecture-aware parameter diff (including tied weights and adapter coverage).
4. Behavioral probe results on versioned retain, forget, safety, robustness and task suites.
5. Performance/resource regression results and deployment compatibility.
6. Signed provenance, change ticket, authorized requester and independent approver.
7. Candidate artifact location, rollback reference and final promotion decision.

**Unlearning is not the same as runtime forgetting.** An LSTM forget gate can reduce the contribution of prior cell state at a time step; that does not erase information encoded in learned parameters. Model unlearning requires separate forget-set and retain-set evaluation, generalization probes, safety regression testing, and comparison with a reference model trained without the forget data where feasible.

Do not claim exact erasure from a low score on one prompt or probe set. Report the scope and limitations of the evidence.

In [ ]:
def model_change_manifest(source_model: nn.Module, candidate_model: nn.Module,
                          adapter_name: str, adapter_version: str,
                          operation: str, change_ticket: str):
    """Create a deterministic, reviewable manifest; does not approve the change."""
    source_state = source_model.state_dict()
    candidate_state = candidate_model.state_dict()
    if set(source_state) != set(candidate_state):
        raise ValueError("State-dict keys differ; require an explicit migration adapter")
    changed = []
    for name in sorted(source_state):
        a, b = source_state[name].detach().cpu(), candidate_state[name].detach().cpu()
        if a.shape != b.shape or a.dtype != b.dtype:
            changed.append({"name": name, "shape_before": list(a.shape),
                            "shape_after": list(b.shape), "dtype_changed": str(a.dtype) != str(b.dtype)})
        elif not torch.equal(a, b):
            delta = (b.float() - a.float())
            changed.append({"name": name, "shape": list(a.shape),
                            "l2_delta": float(torch.linalg.vector_norm(delta)),
                            "max_abs_delta": float(delta.abs().max()) if delta.numel() else 0.0})
    return {
        "operation": operation,
        "change_ticket": change_ticket,
        "adapter": {"name": adapter_name, "version": adapter_version},
        "source_sha256": state_dict_sha256(source_model),
        "candidate_sha256": state_dict_sha256(candidate_model),
        "changed_tensor_count": len(changed),
        "changed_tensors": changed,
        "promotion_status": "not_authorized",
        "created_at_utc": utc_now_iso(),
    }

# Demonstrate a read-only comparison against an isolated candidate.
_candidate = copy.deepcopy(model)
with torch.no_grad():
    _candidate.blocks[0].attn.o_proj.weight[0, 0].add_(1e-4)
_demo_manifest = model_change_manifest(
    model, _candidate, adapter_name="MiniDecoderTransformer",
    adapter_version="0.1", operation="controlled_parameter_perturbation",
    change_ticket="WSP-DEMO-2"
)
assert _demo_manifest["changed_tensor_count"] >= 1
assert _demo_manifest["source_sha256"] == state_dict_sha256(model)
print("Candidate manifest created; source checkpoint remains unchanged.")
print("Changed tensors:", _demo_manifest["changed_tensor_count"])

## 17. LSTM memory-gate adapter: inspect retention pathways, not individual facts

LSTMs provide an explicit, useful test fixture for studying memory control. Their sigmoid gates regulate the cell-state pathway, while tanh generates candidate content. Gate activations are dynamic values conditioned on inputs and hidden state; the gate weight matrices do not map one-to-one to human-readable facts.

The following adapter exposes gate activations for a small PyTorch LSTM. It is an analysis fixture, not a general-purpose reader for every LSTM checkpoint layout.

In [ ]:
class InspectableLSTMCell(nn.Module):
    """Single-layer LSTM cell exposing gates for controlled experiments."""
    def __init__(self, input_size=8, hidden_size=12):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.x_proj = nn.Linear(input_size, 4 * hidden_size)
        self.h_proj = nn.Linear(hidden_size, 4 * hidden_size, bias=False)

    def forward(self, x, state):
        h_prev, c_prev = state
        gates = self.x_proj(x) + self.h_proj(h_prev)
        i_raw, f_raw, g_raw, o_raw = gates.chunk(4, dim=-1)
        i_gate = torch.sigmoid(i_raw)
        f_gate = torch.sigmoid(f_raw)
        g_candidate = torch.tanh(g_raw)
        o_gate = torch.sigmoid(o_raw)
        c_next = f_gate * c_prev + i_gate * g_candidate
        h_next = o_gate * torch.tanh(c_next)
        return h_next, c_next, {"input": i_gate, "forget": f_gate,
                                "candidate": g_candidate, "output": o_gate}


def summarize_lstm_gates(gate_history):
    summary = {}
    for name in ("input", "forget", "candidate", "output"):
        values = torch.cat([step[name].detach().reshape(-1) for step in gate_history])
        summary[name] = {
            "mean": float(values.mean()),
            "std": float(values.std(unbiased=False)),
            "near_zero_fraction": float((values.abs() < 0.05).float().mean()),
            "near_one_fraction": float((values > 0.95).float().mean()) if name != "candidate" else None,
        }
    return summary


torch.manual_seed(23)
lstm_cell = InspectableLSTMCell().to(DEVICE).eval()
batch_size, steps = 4, 10
h = torch.zeros(batch_size, lstm_cell.hidden_size, device=DEVICE)
c = torch.zeros_like(h)
gate_history = []
with torch.no_grad():
    for _ in range(steps):
        x_t = torch.randn(batch_size, lstm_cell.input_size, device=DEVICE)
        h, c, gates = lstm_cell(x_t, (h, c))
        gate_history.append(gates)

lstm_gate_summary = summarize_lstm_gates(gate_history)
print(json.dumps(lstm_gate_summary, indent=2))

### LSTM unlearning experiment protocol

Use synthetic sequence classes with disjoint retain and forget patterns. Train a baseline, preserve an immutable checkpoint, and create an isolated candidate. Compare at least:

- baseline performance on retain and forget sets;
- candidate performance on retain and forget sets;
- paraphrase/variation and sequence-context probes;
- gate activation distributions and cell-state trajectories;
- parameter deltas and checkpoint hashes;
- safety and unrelated-task regressions;
- a reference model trained without the forget examples, when feasible.

A gate-bias intervention is an experiment about runtime memory dynamics, not proof of model-level unlearning. Gate-level weight changes can affect many sequences and should be evaluated for collateral damage. The experiment must fail closed if the architecture adapter cannot identify the gate layout reliably.

## 18. Application modernization and AI-native lifecycle integration

The Weight Space Programming engine is intended to support an AI-enabled application lifecycle, not replace the application SDLC. A Python service (for example, FastAPI) can expose typed operations to an optional Angular control-plane UI. The service orchestrates model inspection and assessment; the UI presents experiment status, evidence, policy decisions and approvals.

### Lifecycle control points

| Lifecycle phase | Application deliverable | Parallel security deliverable |
|---|---|---|
| Discover / assess | Legacy inventory, dependencies, data flows | Threat model, attack-surface inventory, initial risk register |
| Modernize / digitalize | API boundaries, service decomposition, data migration | AuthN/AuthZ design, privacy controls, secure migration validation |
| Build | Source, tests, SDKs, model integration | SAST, SCA, secrets, license and code policy checks |
| Containerize / provision | OCI image, Terraform, Kubernetes manifests | Image/SBOM scans, IaC policies, admission and least-privilege checks |
| Integrate AI | Model registry, RAG, agents, inference API | Model provenance, weight integrity, prompt/tool boundaries, data controls |
| Release | Versioned application and model artifacts | Signed attestations, independent approval, deployment gate |
| Operate / scale | Runtime, autoscaling, updates, observability | Runtime detection, identity/network posture, event-triggered reassessment |
| Change / unlearn | Fine-tuning, parameter edits, knowledge-base updates | Change manifest, retain/forget/safety evaluation, signed authorization |
| End of life | Decommission services and models | Revoke identities, disable endpoints, disposition data, archive audit evidence |

### Stable service boundary (design target)

- `inspect_artifact(...)`
- `fingerprint_model(...)`
- `compare_models(...)`
- `run_probe_suite(...)`
- `analyze_sensitivity(...)`
- `create_model_change_manifest(...)`
- `assess_security_posture(...)`
- `evaluate_zero_trust_gate(...)`
- `evaluate_unlearning(...)`
- `export_evidence_report(...)`

The UI/API must not directly mutate production weights. Candidate editing runs in an isolated worker against a copied artifact; a separate authorization service controls promotion.

## 19. Updated engineering best practices and research boundaries

1. **One shared engine, architecture-aware adapters.** Keep the existing Transformer as the primary demonstration; add LSTM as a complementary adapter rather than replacing the Transformer or splitting the project into more notebooks.
2. **Read-only by default.** Original checkpoints are immutable; candidate changes are isolated, hashed and independently reviewed.
3. **Behavior is measured, not inferred from one tensor.** Weight statistics and anomalies are triage signals. Claims about knowledge, perspective, safety or unlearning require behavioral and causal evaluation.
4. **Unlearning is first-class.** Evaluate forget efficacy, retain quality, generalization, safety regressions and reference-model distance. Runtime gate forgetting is not model unlearning.
5. **Security is parallel to the SDLC.** Every artifact and lifecycle phase has a threat model, attack-surface inventory, security checks, policy gates and audit evidence.
6. **Event-driven reassessment.** Material source, dependency, infrastructure, container, model, weight, data, permission and runtime changes invalidate or refresh relevant evidence.
7. **Zero-trust promotion.** Missing, stale, mismatched or unauthorized evidence quarantines the candidate. No single scanner or score establishes that an artifact is vulnerability-free.
8. **End-of-life is in scope.** Revoke credentials, disable services, dispose of data and artifacts appropriately, and preserve required audit evidence.
9. **Research maturity is explicit.** Label work as reproducible baseline, research reproduction, prototype or roadmap. Do not represent prototypes as validated methods.

### Project structure remains three notebooks

- `01_weight_space_research_lab.ipynb` — mathematical and research foundations.
- `02_weight_space_programming_v3_security_native.ipynb` — central engine, programmable weight analysis, LSTM/unlearning experiments, and parallel zero-trust SDLC assurance.
- `03_counterfactual_optimization_evaluation.ipynb` — candidate evaluation, optimization, and promotion experiments.

This is an update to the central notebook, not a new notebook series.

## 20. Executable security-tool integration layer

This section turns the security-native SDLC design into an integration scaffold for real scanners. It supports **Semgrep (SAST), pip-audit (Python SCA), Checkov (IaC), Trivy (filesystem/container scanning), and Gitleaks (secret detection)**.

Safety properties:
- Scanner execution is opt-in; merely running the notebook does not scan a repository.
- Commands use argument arrays (never `shell=True`) and bounded timeouts.
- Tool availability, exit status, duration, output hash, and a redacted summary are recorded.
- Raw scanner output is not printed or placed in the notebook because secret scanners can return sensitive values.
- A missing scanner is reported as `NOT_RUN`, never as a pass.
- Scanner findings are evidence for triage; they do not prove an application is vulnerability-free.

Install the tools in an isolated, pinned development environment and review each tool's license and version before enabling it.

In [ ]:
import os, shutil, subprocess, time, hashlib, json
from pathlib import Path
from datetime import datetime, timezone

SCANNER_TIMEOUT_SECONDS = 600


def _sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8", errors="replace")).hexdigest()


def scanner_inventory():
    """Report scanner availability without executing a scan."""
    commands = {"semgrep": "semgrep", "pip_audit": "pip-audit",
                "checkov": "checkov", "trivy": "trivy", "gitleaks": "gitleaks"}
    return {name: shutil.which(binary) for name, binary in commands.items()}


def build_scanner_commands(repo_path, requirements_path=None):
    """Build read-only scanner commands; callers explicitly choose which to run."""
    repo = str(Path(repo_path).resolve())
    commands = {
        "semgrep": ["semgrep", "scan", "--config", "auto", "--json", repo],
        "checkov": ["checkov", "-d", repo, "-o", "json"],
        "trivy_fs": ["trivy", "fs", "--format", "json", repo],
        "gitleaks": ["gitleaks", "dir", "--report-format", "json", "--no-banner", repo],
    }
    if requirements_path:
        req = str(Path(requirements_path).resolve())
        commands["pip_audit"] = ["pip-audit", "--format", "json", "-r", req]
    return commands


def _json_finding_count(raw):
    """Best-effort count only; never return raw scanner content."""
    try:
        obj = json.loads(raw)
    except Exception:
        return None
    if isinstance(obj, list):
        return len(obj)
    if not isinstance(obj, dict):
        return None
    for key in ("results", "Results", "vulnerabilities", "Vulnerabilities", "findings", "Findings"):
        value = obj.get(key)
        if isinstance(value, list):
            return len(value)
    # Semgrep uses results; Trivy commonly uses Results grouped by target.
    if isinstance(obj.get("Results"), list):
        return sum(len(x.get("Vulnerabilities") or []) + len(x.get("Misconfigurations") or [])
                   + len(x.get("Secrets") or []) for x in obj["Results"] if isinstance(x, dict))
    return None


def run_security_scanners(repo_path, selected, requirements_path=None, timeout=SCANNER_TIMEOUT_SECONDS):
    """Run only explicitly selected scanners; return metadata, not raw findings."""
    allowed = {"semgrep", "pip_audit", "checkov", "trivy_fs", "gitleaks"}
    unknown = set(selected) - allowed
    if unknown:
        raise ValueError(f"Unsupported scanner names: {sorted(unknown)}")
    commands = build_scanner_commands(repo_path, requirements_path)
    inventory = scanner_inventory()
    records = []
    for name in selected:
        binary_key = "trivy" if name == "trivy_fs" else name
        if not inventory.get(binary_key):
            records.append({"scanner": name, "status": "NOT_RUN", "reason": "executable_not_found"})
            continue
        if name not in commands:
            records.append({"scanner": name, "status": "NOT_RUN", "reason": "required_input_missing"})
            continue
        started = time.monotonic()
        try:
            proc = subprocess.run(commands[name], capture_output=True, text=True,
                                 timeout=timeout, check=False, shell=False)
            elapsed = round(time.monotonic() - started, 3)
            raw = (proc.stdout or "") + (proc.stderr or "")
            records.append({"scanner": name,
                            "status": "COMPLETED" if proc.returncode in (0, 1) else "ERROR",
                            "return_code": proc.returncode,
                            "duration_seconds": elapsed,
                            "output_sha256": _sha256_text(raw),
                            "finding_count_estimate": _json_finding_count(proc.stdout or ""),
                            "raw_output_persisted": False})
        except subprocess.TimeoutExpired:
            records.append({"scanner": name, "status": "TIMEOUT",
                            "duration_seconds": round(time.monotonic() - started, 3),
                            "raw_output_persisted": False})
        except Exception as exc:
            records.append({"scanner": name, "status": "ERROR",
                            "error_type": type(exc).__name__,
                            "raw_output_persisted": False})
    return {"timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "repo_path": str(Path(repo_path).resolve()),
            "scanner_inventory": {k: bool(v) for k, v in inventory.items()},
            "results": records,
            "interpretation": "NOT_RUN and ERROR are not passes; review full findings in the scanner's secured output workflow."}

print("Scanner availability:", scanner_inventory())
print("No scanners have been executed. Set an explicit repository path and selected scanner list to run.")

### Explicit scan invocation (operator-controlled)

Set `RUN_SECURITY_SCANS = True` only after reviewing the target repository, scanner versions, permissions, and output-handling requirements. Use a read-only checkout where possible. Do not point secret-scanning output at a publicly shared location.

In [ ]:
# Example: intentionally disabled by default.
RUN_SECURITY_SCANS = False
REPOSITORY_TO_ASSESS = Path(".").resolve()
REQUIREMENTS_FILE = REPOSITORY_TO_ASSESS / "requirements.txt"
SELECTED_SCANNERS = ["semgrep", "pip_audit", "checkov", "trivy_fs", "gitleaks"]

if RUN_SECURITY_SCANS:
    scanner_report = run_security_scanners(
        REPOSITORY_TO_ASSESS,
        SELECTED_SCANNERS,
        requirements_path=REQUIREMENTS_FILE if REQUIREMENTS_FILE.exists() else None,
    )
    print(json.dumps(scanner_report, indent=2))
else:
    print("Security scans disabled. No repository scan was performed.")

## 21. Change-triggered reassessment and lifecycle events

Security checks should run when the attack surface changes, not only on a calendar. This lightweight event router classifies change events and maps them to required reassessment domains. In production, connect it to GitHub/GitLab webhooks, artifact-registry events, Kubernetes admission events, and model-registry events.

The router does not grant deployment approval. It creates an assessment plan; the independent release gate still decides whether evidence is complete and current.

In [ ]:
from dataclasses import dataclass
from typing import Tuple

EVENT_TO_CHECKS = {
    "source_change": ("source_sast", "unit_tests", "secret_scan", "api_regression"),
    "dependency_change": ("dependency_sca", "license_review", "sbom_refresh"),
    "container_rebuild": ("container_scan", "sbom_refresh", "image_signature"),
    "iac_change": ("iac_scan", "policy_tests", "privilege_review"),
    "api_change": ("api_security_tests", "authz_tests", "contract_tests"),
    "model_checkpoint_change": ("checkpoint_provenance", "weight_diff", "behavioral_regression", "safety_eval"),
    "fine_tuning": ("training_data_provenance", "weight_diff", "retention_eval", "safety_eval"),
    "unlearning": ("signed_unlearning_request", "forget_eval", "retain_eval", "recovery_eval", "checkpoint_integrity"),
    "agent_tool_change": ("tool_inventory", "least_privilege_review", "prompt_injection_eval", "action_authorization_tests"),
    "runtime_config_change": ("config_policy", "runtime_regression", "access_review"),
    "end_of_life": ("decommission_authorization", "credential_revocation", "data_retention_review", "artifact_archival"),
}

@dataclass(frozen=True)
class AssessmentPlan:
    event_id: str
    event_type: str
    artifact_id: str
    required_checks: Tuple[str, ...]
    created_at_utc: str
    status: str = "ASSESSMENT_REQUIRED"


def plan_reassessment(event_id, event_type, artifact_id):
    if event_type not in EVENT_TO_CHECKS:
        raise ValueError(f"Unknown lifecycle event: {event_type}")
    if not event_id or not artifact_id:
        raise ValueError("event_id and artifact_id are required")
    return AssessmentPlan(event_id, event_type, artifact_id,
                          EVENT_TO_CHECKS[event_type], datetime.now(timezone.utc).isoformat())

example_plan = plan_reassessment("evt-demo-001", "model_checkpoint_change", "model://candidate-v2")
print(example_plan)

## 22. Policy-as-code: release gate example

The following Rego policy is a starter policy for a CI/CD gate. It fails closed when required checks are absent, stale, failed, or not tied to the artifact digest. Production policies should also validate signer identity, attestation issuer, exception expiry, environment, and separation of duties.

In [ ]:
REGO_RELEASE_POLICY = "package wsp.release\n\ndefault allow := false\n\nrequired_checks := {\n  \"source_sast\", \"dependency_sca\", \"secret_scan\", \"iac_scan\",\n  \"container_scan\", \"artifact_signature\", \"provenance\",\n  \"behavioral_regression\", \"model_integrity\"\n}\n\nvalid_check(name) if {\n  some check in input.checks\n  check.name == name\n  check.status == \"PASS\"\n  check.artifact_digest == input.artifact_digest\n  check.fresh == true\n}\n\nmissing_checks := required_checks - {name | some name in required_checks; valid_check(name)}\n\nallow if {\n  input.authorized == true\n  input.artifact_digest != \"\"\n  count(missing_checks) == 0\n  input.independent_approval == true\n}\n"
print(REGO_RELEASE_POLICY)

## 23. Secure unlearning request contract

Unlearning is a privileged model change. A request must identify the model digest, scope, legal/business authority, forget and retain evaluation suites, requester, approver, and rollback artifact. The validator below checks request completeness; it does **not** perform unlearning or prove that a model has forgotten information.

In [ ]:
UNLEARNING_REQUIRED_FIELDS = {
    "request_id", "model_digest", "scope_id", "authority_reference",
    "forget_eval_version", "retain_eval_version", "requester", "independent_approver",
    "rollback_checkpoint_digest", "method_id", "change_ticket"
}

def validate_unlearning_request(request):
    missing = sorted(k for k in UNLEARNING_REQUIRED_FIELDS if not request.get(k))
    if request.get("requester") == request.get("independent_approver"):
        missing.append("independent_approver_must_differ_from_requester")
    digest = request.get("model_digest", "")
    rollback = request.get("rollback_checkpoint_digest", "")
    if digest and (len(digest) != 64 or any(c not in "0123456789abcdef" for c in digest.lower())):
        missing.append("model_digest_must_be_sha256")
    if rollback and (len(rollback) != 64 or any(c not in "0123456789abcdef" for c in rollback.lower())):
        missing.append("rollback_checkpoint_digest_must_be_sha256")
    return {"valid": not missing, "missing_or_invalid": missing,
            "execution_authorized": False,
            "note": "Completeness validation only; a separate approved workflow is required."}

assert validate_unlearning_request({})["valid"] is False
print("Unlearning request contract checks passed.")

## 24. CI/CD workflow template

This GitHub Actions template demonstrates parallel application and security jobs. Replace pinned action versions, runner labels, paths, and organization-specific policies after review. The example intentionally blocks release when a required security job fails. It is a template, not a claim that these tools alone establish zero vulnerabilities.

In [ ]:
GITHUB_ACTIONS_WORKFLOW = "name: application-and-model-assurance\non:\n  pull_request:\n  push:\n    branches: [main]\n\npermissions:\n  contents: read\n\njobs:\n  application-tests:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: \"3.11\"\n      - run: pip install -r requirements.txt\n      - run: python -m pytest -q\n\n  security-assessment:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: \"3.11\"\n      - run: pip install semgrep pip-audit checkov\n      - run: semgrep scan --config auto .\n      - run: pip-audit -r requirements.txt\n      - run: checkov -d .\n\n  release-authorization:\n    needs: [application-tests, security-assessment]\n    if: github.event_name == 'push' && github.ref == 'refs/heads/main'\n    runs-on: ubuntu-latest\n    steps:\n      - run: echo \"Promotion requires verified attestations and independent approval.\"\n"
print(GITHUB_ACTIONS_WORKFLOW)

## 25. Operational boundaries and acceptance criteria

- **No zero-vulnerability guarantee:** scanners have coverage limits, false positives, false negatives, and configuration dependencies. State residual risk explicitly.
- **No automatic production weight edits:** all candidate transformations occur on isolated copies and require evaluation, approval, signing, and rollback readiness.
- **No fact-to-weight assumptions:** parameters are distributed representations; weight inspection does not identify a discrete fact without validated causal evidence.
- **No unlearning overclaim:** test forget efficacy, retain utility, paraphrase/generalization, extraction/recovery, and compare against a retrained reference where feasible.
- **No self-approval:** the requester or transformation service cannot independently approve its own production promotion.
- **Lifecycle completeness:** include modernization, build, release, runtime, model updates, incident response, and end-of-life decommissioning.

The engineering objective is risk reduction and continuous evidence—not a claim that an application or model can ever be guaranteed vulnerability-free.